In [1]:
import pandas as pd
import numpy as np

## Loading data

In [2]:
attotaed_df = pd.read_json("./data/raw/dev_labeled_ChatGPT.jsonl", lines=True)
attotaed_df.head()

,input,output,topic,cat,annotations
0,Question: Tell me a bio of Marianne McAndrew.,Marianne McAndrew is an American actress and s...,Marianne McAndrew,"[very rare, North America]",[{'text': 'Marianne McAndrew is an American ac...
1,Question: Tell me a bio of Gerhard Fischer (in...,Gerhard Fischer is an inventor and entrepreneu...,Gerhard Fischer (inventor),"[very rare, North America]",[{'text': 'Gerhard Fischer is an inventor and ...
2,Question: Tell me a bio of Joey D. Vieira.,Joey D. Vieira is an American actor and stunt ...,Joey D. Vieira,"[very rare, North America]",[{'text': 'Joey D. Vieira is an American actor...
3,Question: Tell me a bio of Taral Hicks.,"Taral Hicks is an American actress and singer,...",Taral Hicks,"[very rare, North America]","[{'text': 'She later transitioned to acting, a..."
4,Question: Tell me a bio of Vincenzo Tusa.,Vincenzo Tusa is an Italian politician who was...,Vincenzo Tusa,"[very rare, Europe/Middle East]",[{'text': 'He is a member of the Democratic Pa...


In [3]:
df = attotaed_df.explode('annotations',ignore_index=True)
print(f"pre shape: {attotaed_df.shape}")
print(f"post shape: {df.shape}")

pre shape: (31, 5)
post shape: (52, 5)


In [4]:
df['input'] = df['input'].apply(lambda x: x.replace("Question: ", ""))

## Flattan the annototaion column in to seperate columns

In [5]:
df['annotations'].shape

(52,)

In [6]:
annotations = pd.json_normalize(df['annotations'])
print(f"annotations shape: {annotations.shape}")
print(f"annotations columns: {annotations.columns}")

annotations shape: (52, 4)
annotations columns: Index(['text', 'is-relevant', 'model-atomic-facts', 'human-atomic-facts'], dtype='object')


# Trying to remove unrelevant data

In [7]:
annotations[annotations["is-relevant"] == True].shape

(52, 4)

# converting json data in to tabuler format

In [8]:
print(f"annotations shape: {annotations.shape}")
print(f"df shape: {df.shape}")

annotations shape: (52, 4)
df shape: (52, 5)


In [9]:
df = pd.concat([df.drop(columns=["annotations"], inplace=False), annotations], axis=1)
print(f"df shape: {df.shape}")
print(f"df columns: {df.columns}")

df shape: (52, 8)
df columns: Index(['input', 'output', 'topic', 'cat', 'text', 'is-relevant',
       'model-atomic-facts', 'human-atomic-facts'],
      dtype='object')


In [10]:
df[:1]

,input,output,topic,cat,text,is-relevant,model-atomic-facts,human-atomic-facts
0,Tell me a bio of Marianne McAndrew.,Marianne McAndrew is an American actress and s...,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,True,[{'text': 'Marianne McAndrew is an American ac...,"[{'text': 'Marianne McAndrew is an American.',..."


In [11]:
print(f"pre shape: {df.shape}")

df = df.explode('human-atomic-facts',ignore_index=True)

print(f"post shape: {df.shape}")


pre shape: (52, 8)
post shape: (221, 8)


In [12]:
df[:1]

,input,output,topic,cat,text,is-relevant,model-atomic-facts,human-atomic-facts
0,Tell me a bio of Marianne McAndrew.,Marianne McAndrew is an American actress and s...,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,True,[{'text': 'Marianne McAndrew is an American ac...,"{'text': 'Marianne McAndrew is an American.', ..."


In [13]:
human_facts = pd.json_normalize(df['human-atomic-facts'])
human_facts.rename(columns={'text': "atomic-fact"}, inplace=True)

print(f"post shape: {human_facts.shape}")
print(f"post columns: {human_facts.columns}")
human_facts.head(5)

post shape: (221, 2)
post columns: Index(['atomic-fact', 'label'], dtype='object')


,atomic-fact,label
0,Marianne McAndrew is an American.,NS
1,Marianne McAndrew is an actress.,S
2,Marianne McAndrew is a singer.,NS
3,"Marianne McAndrew was born on November 21, 1942.",NS
4,Marianne McAndrew was born in Cleveland.,NS


In [14]:
print(f"post shape: {df.shape}")
print(f"post columns: {df.columns}")

df = df.reset_index(drop=True)
human_facts = human_facts.reset_index(drop=True)
df = pd.concat([df, human_facts], axis=1)

print(f"post shape: {df.shape}")
print(f"post columns: {df.columns}")

df.head(5)

post shape: (221, 8)
post columns: Index(['input', 'output', 'topic', 'cat', 'text', 'is-relevant',
       'model-atomic-facts', 'human-atomic-facts'],
      dtype='object')
post shape: (221, 10)
post columns: Index(['input', 'output', 'topic', 'cat', 'text', 'is-relevant',
       'model-atomic-facts', 'human-atomic-facts', 'atomic-fact', 'label'],
      dtype='object')


,input,output,topic,cat,text,is-relevant,model-atomic-facts,human-atomic-facts,atomic-fact,label
0,Tell me a bio of Marianne McAndrew.,Marianne McAndrew is an American actress and s...,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,True,[{'text': 'Marianne McAndrew is an American ac...,"{'text': 'Marianne McAndrew is an American.', ...",Marianne McAndrew is an American.,NS
1,Tell me a bio of Marianne McAndrew.,Marianne McAndrew is an American actress and s...,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,True,[{'text': 'Marianne McAndrew is an American ac...,"{'text': 'Marianne McAndrew is an actress.', '...",Marianne McAndrew is an actress.,S
2,Tell me a bio of Marianne McAndrew.,Marianne McAndrew is an American actress and s...,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,True,[{'text': 'Marianne McAndrew is an American ac...,"{'text': 'Marianne McAndrew is a singer.', 'la...",Marianne McAndrew is a singer.,NS
3,Tell me a bio of Marianne McAndrew.,Marianne McAndrew is an American actress and s...,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,True,[{'text': 'Marianne McAndrew is an American ac...,{'text': 'Marianne McAndrew was born on Novemb...,"Marianne McAndrew was born on November 21, 1942.",NS
4,Tell me a bio of Marianne McAndrew.,Marianne McAndrew is an American actress and s...,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,True,[{'text': 'Marianne McAndrew is an American ac...,{'text': 'Marianne McAndrew was born in Clevel...,Marianne McAndrew was born in Cleveland.,NS


In [15]:
df.drop(columns=["output","is-relevant","model-atomic-facts","human-atomic-facts"], inplace=True)
df.columns

Index(['input', 'topic', 'cat', 'text', 'atomic-fact', 'label'], dtype='object')

In [16]:
df[df.duplicated(["topic", "atomic-fact"], keep=False)].sort_values("atomic-fact")[["topic", "atomic-fact", "label"]]

,topic,atomic-fact,label


In [17]:
df['label'].value_counts()

label
S     112
NS     93
IR     16
Name: count, dtype: int64

In [18]:
df["label"] = df["label"].replace("IR", "NS")
df["label"].value_counts()

label
S     112
NS    109
Name: count, dtype: int64

In [19]:
attotaed_df = df
print(f"attotaed_df shape: {attotaed_df.shape}")
attotaed_df.head(10)

attotaed_df shape: (221, 6)


,input,topic,cat,text,atomic-fact,label
0,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew is an American.,NS
1,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew is an actress.,S
2,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew is a singer.,NS
3,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,"Marianne McAndrew was born on November 21, 1942.",NS
4,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew was born in Cleveland.,NS
5,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Cleveland is in Ohio.,NS
6,Tell me a bio of Gerhard Fischer (inventor).,Gerhard Fischer (inventor),"[very rare, North America]",Gerhard Fischer is an inventor and entrepreneu...,Gerhard Fischer is an inventor.,S
7,Tell me a bio of Gerhard Fischer (inventor).,Gerhard Fischer (inventor),"[very rare, North America]",Gerhard Fischer is an inventor and entrepreneu...,Gerhard Fischer is an entrepreneur.,S
8,Tell me a bio of Gerhard Fischer (inventor).,Gerhard Fischer (inventor),"[very rare, North America]",Gerhard Fischer is an inventor and entrepreneu...,Gerhard Fischer is best known for inventing a ...,S
9,Tell me a bio of Gerhard Fischer (inventor).,Gerhard Fischer (inventor),"[very rare, North America]",Gerhard Fischer is an inventor and entrepreneu...,"Gerhard Fischer invented the first handheld, b...",S


In [20]:
df = pd.read_json("./data/raw/passages_bm25_ChatGPT_humfacts.jsonl", lines=True)
print(f"shape: {df.shape}")
df.head(5)

shape: (5427, 3)


,name,sent,passages
0,Lanny Flaherty,Lanny Flaherty is an American.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny..."
1,Lanny Flaherty,Lanny Flaherty is an actor.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny..."
2,Lanny Flaherty,"Lanny Flaherty was born on December 18, 1949.","[{'title': 'Lanny Flaherty', 'text': '<s>Lanny..."
3,Lanny Flaherty,Lanny Flaherty was born in Pensacola.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny..."
4,Lanny Flaherty,Pensacola is in Florida.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny..."


In [21]:
df["passage_list"] = df["passages"].apply(lambda ps: [p["text"] for p in ps])
print(f"passage_text shape: {df['passage_list'].shape}")
df["passage_list"].head(5)
df.head(5)

passage_text shape: (5427,)


,name,sent,passages,passage_list
0,Lanny Flaherty,Lanny Flaherty is an American.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
1,Lanny Flaherty,Lanny Flaherty is an actor.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
2,Lanny Flaherty,"Lanny Flaherty was born on December 18, 1949.","[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
3,Lanny Flaherty,Lanny Flaherty was born in Pensacola.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
4,Lanny Flaherty,Pensacola is in Florida.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...


In [22]:
df.head(10)

,name,sent,passages,passage_list
0,Lanny Flaherty,Lanny Flaherty is an American.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
1,Lanny Flaherty,Lanny Flaherty is an actor.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
2,Lanny Flaherty,"Lanny Flaherty was born on December 18, 1949.","[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
3,Lanny Flaherty,Lanny Flaherty was born in Pensacola.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
4,Lanny Flaherty,Pensacola is in Florida.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
5,Lanny Flaherty,He has appeared in numerous films.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
6,Lanny Flaherty,He has appeared in numerous television shows.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
7,Lanny Flaherty,He has appeared in numerous theater productions.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
8,Lanny Flaherty,His career began in the late 1970s.,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...
9,Lanny Flaherty,His notable film credits include King of New Y...,"[{'title': 'Lanny Flaherty', 'text': '<s>Lanny...",[<s>Lanny Flaherty Lanny Flaherty (born July 2...


In [23]:
print(f"pre-merged shape: {df.shape}")
print(f"pre-merged columns: {df.columns}")

df = pd.merge(attotaed_df, df, left_on="atomic-fact", right_on="sent", how="left")

print(f"post-merged shape: {df.shape}")
print(f"post-merged columns: {df.columns}")
df.head(5)

pre-merged shape: (5427, 4)
pre-merged columns: Index(['name', 'sent', 'passages', 'passage_list'], dtype='object')
post-merged shape: (225, 10)
post-merged columns: Index(['input', 'topic', 'cat', 'text', 'atomic-fact', 'label', 'name', 'sent',
       'passages', 'passage_list'],
      dtype='object')


,input,topic,cat,text,atomic-fact,label,name,sent,passages,passage_list
0,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew is an American.,NS,Marianne McAndrew,Marianne McAndrew is an American.,"[{'title': 'Marianne McAndrew', 'text': '<s>Ma...",[<s>Marianne McAndrew Marianne Christine McAnd...
1,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew is an actress.,S,Marianne McAndrew,Marianne McAndrew is an actress.,"[{'title': 'Marianne McAndrew', 'text': '<s>Ma...",[<s>Marianne McAndrew Marianne Christine McAnd...
2,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew is a singer.,NS,Marianne McAndrew,Marianne McAndrew is a singer.,"[{'title': 'Marianne McAndrew', 'text': '<s>Ma...",[<s>Marianne McAndrew Marianne Christine McAnd...
3,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,"Marianne McAndrew was born on November 21, 1942.",NS,Marianne McAndrew,"Marianne McAndrew was born on November 21, 1942.","[{'title': 'Marianne McAndrew', 'text': '<s>Ma...",[<s>Marianne McAndrew Marianne Christine McAnd...
4,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew was born in Cleveland.,NS,Marianne McAndrew,Marianne McAndrew was born in Cleveland.,"[{'title': 'Marianne McAndrew', 'text': '<s>Ma...",[<s>Marianne McAndrew Marianne Christine McAnd...


In [24]:
df.drop(columns=["name", "sent","passages",], inplace=True)

In [25]:
df["passage_list"]=df["passage_list"].apply(tuple) # covert list to tuple for hashing becuase list is unhashable and cannot be used in duplicated() function

In [26]:
mask = df.duplicated(subset=["topic", "atomic-fact","passage_list"], keep=False)
df[mask].sort_values("atomic-fact")[["topic", "atomic-fact", "label"]]

,topic,atomic-fact,label


In [27]:
df.rename(columns={"passage_list": "reference_passages"}, inplace=True)
df.columns

Index(['input', 'topic', 'cat', 'text', 'atomic-fact', 'label',
       'reference_passages'],
      dtype='object')

In [28]:
print(f"Number of null values in each column:\n{df.isnull().sum()}")
print(f"Number of empty lists in reference_passages column:\n{df['reference_passages'].apply(lambda x: isinstance(x,list) and len(x)==0).sum()}")

Number of null values in each column:
input                 0
topic                 0
cat                   0
text                  0
atomic-fact           0
label                 0
reference_passages    0
dtype: int64
Number of empty lists in reference_passages column:
0


In [29]:
df

,input,topic,cat,text,atomic-fact,label,reference_passages
0,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew is an American.,NS,(<s>Marianne McAndrew Marianne Christine McAnd...
1,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew is an actress.,S,(<s>Marianne McAndrew Marianne Christine McAnd...
2,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew is a singer.,NS,(<s>Marianne McAndrew Marianne Christine McAnd...
3,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,"Marianne McAndrew was born on November 21, 1942.",NS,(<s>Marianne McAndrew Marianne Christine McAnd...
4,Tell me a bio of Marianne McAndrew.,Marianne McAndrew,"[very rare, North America]",Marianne McAndrew is an American actress and s...,Marianne McAndrew was born in Cleveland.,NS,(<s>Marianne McAndrew Marianne Christine McAnd...
...,...,...,...,...,...,...,...
220,Tell me a bio of Ji Sung.,Ji Sung,"[very freq, Asia/Pacific]",He has since then starred in numerous popular ...,"-""Protect the Boss."" is popular.",S,(<s>Ji Sung Ji Sung (born Kwak Tae-geun on 27 ...
221,Tell me a bio of Ji Sung.,Ji Sung,"[very freq, Asia/Pacific]",He has since then starred in numerous popular ...,"He has since then starred in ""Kill Me, Heal Me."".",S,"(odrama ""Secret Love"". He said he chose the dr..."
222,Tell me a bio of Ji Sung.,Ji Sung,"[very freq, Asia/Pacific]",He has since then starred in numerous popular ...,"""Kill Me, Heal Me."" is popular.",S,(<s>Ji Sung Ji Sung (born Kwak Tae-geun on 27 ...
223,Tell me a bio of Ji Sung.,Ji Sung,"[very freq, Asia/Pacific]",He has since then starred in numerous popular ...,"He has since then starred in ""Familiar Wife."".",S,"(odrama ""Secret Love"". He said he chose the dr..."
